# Look at mapping breadth and depth of 100 x metagenomes to singleclust genes

In [10]:
import polars as pl
import glob
import os

## Read in individual metag x species singleclust depth reports

These files contain depth-of-mapping information for all our genes.

In [11]:
DIR='../outputs.cds/singleclust/bam.bak'
template = '../outputs.cds/singleclust/bam/{metag}.x.{species}.depth.txt'

def read_depth(metag, species):
    filename = template.format(metag=metag, species=species)
    df = pl.read_csv(filename, separator='\t', has_header=False,
                 new_columns=('gene', 'pos', 'cov', 'foo'))
    sum_df = df.group_by('gene').agg(
        ((pl.col("cov") > 0).sum() / pl.col("pos").len()).alias("breadth")
    ).with_columns(
        (pl.lit(metag).alias("metag")),
        (pl.lit(species).alias("species"))
)
    return sum_df

read_depth('ERR1135199', 's__Cryptobacteroides sp900546925')

gene,breadth,metag,species
str,f64,str,str
"""BEKEPKDC_01498""",0.795533,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""OIJDNDLK_01266""",0.64176,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""CPBNAKKL_00778""",0.618519,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""BEKEPKDC_01359""",0.706751,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""EPLMDCHM_00231""",0.760417,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
…,…,…,…
"""ILFIENBF_01329""",0.262055,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""KJEHENCE_00427""",0.14128,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""OIJDNDLK_00513""",0.389653,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"


In [12]:
# Read them all in!
filenames = glob.glob(f"{DIR}/*.depth.txt")

dflist = []
for i, n in enumerate(filenames):
    if i % 100 == 0:
        print(f"{i} of {len(filenames)}")
    n = os.path.basename(n)
    metag, _, species, _ = n.split('.', 3)
    dflist.append(read_depth(metag, species))

depth_df = pl.concat(dflist)

print(f"read {len(filenames)} depth files.")

0 of 1400
100 of 1400
200 of 1400
300 of 1400
400 of 1400
500 of 1400
600 of 1400
700 of 1400
800 of 1400
900 of 1400
1000 of 1400
1100 of 1400
1200 of 1400
1300 of 1400
read 1400 depth files.


In [13]:
depth_df

gene,breadth,metag,species
str,f64,str,str
"""GHILBDOG_00591""",0.78905,"""ERR8314733""","""s__UBA2868 sp004552595"""
"""NBDIFHPL_00838""",0.657193,"""ERR8314733""","""s__UBA2868 sp004552595"""
"""MDNFKAHM_01289""",0.582289,"""ERR8314733""","""s__UBA2868 sp004552595"""
"""EEPGLDGH_01407""",0.823642,"""ERR8314733""","""s__UBA2868 sp004552595"""
"""CPAKOFGF_01213""",0.917526,"""ERR8314733""","""s__UBA2868 sp004552595"""
…,…,…,…
"""PAFBNDJL_01579""",0.0,"""SRR12795793""","""s__Bariatricus sp004560705"""
"""EMNHIHKA_01693""",0.705185,"""SRR12795793""","""s__Bariatricus sp004560705"""
"""OOABBEGM_02263""",0.882353,"""SRR12795793""","""s__Bariatricus sp004560705"""


## Summarize our mapping depth results across all the metagenomes

In [14]:
# require 10% of each gene to be covered by at least one read
BREADTH_CUTOFF = 0.1

In [15]:
# aggregate across all metagenomes;
# calculate fraction of metagenomes for which gene mapping exceeds our breadth cutoff
agg_df = depth_df.group_by(['species', 'gene']).agg(
    ((pl.col("breadth") >= BREADTH_CUTOFF).sum() / pl.col("breadth").len()).alias("f")
)
agg_df

species,gene,f
str,str,f64
"""s__Prevotella sp000434975""","""EEMFJJMB_01353""",0.83
"""s__Prevotella sp000434975""","""PMHDHLGK_00812""",0.43
"""s__Sodaliphilus sp004557565""","""MMKFDMID_01280""",0.9
"""s__Cryptobacteroides sp9005469…","""CPBNAKKL_01491""",0.83
"""s__Sodaliphilus sp004557565""","""CNKENCMK_01602""",0.87
…,…,…
"""s__Gemmiger qucibialis""","""KDIJJGOD_01445""",0.21
"""s__Cryptobacteroides sp9005469…","""LELOEEPG_00778""",0.8
"""s__UBA2868 sp004552595""","""PCFNEHJA_00919""",0.62


In [16]:
# print information out by species
for species in sorted(agg_df['species'].unique()):
    print(species)
    foo_df = agg_df.filter(pl.col("species") == species)
    print(foo_df.sort(by='f').filter(pl.col('f') > 0.8))

s__Bariatricus sp004560705
shape: (15, 3)
┌────────────────────────────┬────────────────┬──────┐
│ species                    ┆ gene           ┆ f    │
│ ---                        ┆ ---            ┆ ---  │
│ str                        ┆ str            ┆ f64  │
╞════════════════════════════╪════════════════╪══════╡
│ s__Bariatricus sp004560705 ┆ ELOEBFJM_01459 ┆ 0.81 │
│ s__Bariatricus sp004560705 ┆ EMNHIHKA_01693 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ LNGHPPMH_00679 ┆ 0.83 │
│ s__Bariatricus sp004560705 ┆ NOACAEKI_00860 ┆ 0.83 │
│ s__Bariatricus sp004560705 ┆ OOABBEGM_02263 ┆ 0.84 │
│ …                          ┆ …              ┆ …    │
│ s__Bariatricus sp004560705 ┆ NDCIOGGG_00214 ┆ 0.89 │
│ s__Bariatricus sp004560705 ┆ AIEIMHOC_00182 ┆ 0.89 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_02301 ┆ 0.93 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_00701 ┆ 0.95 │
│ s__Bariatricus sp004560705 ┆ CHOLCMOC_00863 ┆ 0.97 │
└────────────────────────────┴────────────────┴──────┘
s__Cryptobacteroides sp

## Explore specific gene/species/metag combinations

In [17]:
depth_df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth')

gene,breadth,metag,species
str,f64,str,str


In [18]:
depth_df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth').filter(pl.col("metag") == "ERR1135199")

gene,breadth,metag,species
str,f64,str,str
